# RAG Pipeline Demo

## What this notebook does

We build a full RAG pipeline over **5 arXiv papers** pre-fetched in `../arxivx/papers/`:

```
arxivx/papers/texts/*.txt
        │
        ▼
    [ CHUNK ]  RecursiveCharacterTextSplitter  (500 chars, 100 overlap)
        │
        ▼
    [ EMBED ]  sentence-transformers  all-MiniLM-L6-v2  (local, no API key)
        │
        ▼
    [ STORE ]  ChromaDB  persisted to ./chroma_db/
        │
user query ──▶ embed ──▶ [ RETRIEVE ] top-k chunks
                                  │
                                  ▼
                       [ GENERATE ]  Ollama  llama3.2:3b
                                  │
                                  ▼
                               answer
```


In [ ]:
! uv pip install -q chromadb sentence-transformers langchain-text-splitters

In [ ]:
import json
from pathlib import Path

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

# Paths
ARXIVX_BASE  = Path("../arxivx")
CATALOG_PATH = ARXIVX_BASE / "papers" / "catalog.json"
CHROMA_PATH  = Path("./chroma_db")
COLLECTION   = "arxiv_papers"

# Chunking settings
CHUNK_SIZE    = 500   # ~100-120 tokens
CHUNK_OVERLAP = 50

# How many chunks to retrieve per query
TOP_K = 4

# Ollama local server
OLLAMA_BASE  = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:3b"

## Step 1 — Load Papers

Read `catalog.json` to discover all papers, then load each paper's full extracted text and
its metadata JSON.  The catalog and text files were produced by the
`../arxivx/01_fetch_arxiv_papers*.ipynb` notebooks.

In [ ]:
def load_papers(catalog_path: Path) -> list[dict]:
    catalog = json.loads(catalog_path.read_text())
    papers = []
    for entry in catalog:
        # paths in catalog.json are relative to arxivx/  e.g. "papers/texts/xxx.txt"
        txt_file  = ARXIVX_BASE / entry["txt_path"]
        meta_file = ARXIVX_BASE / entry["meta_path"]
        text      = txt_file.read_text(encoding="utf-8")
        metadata  = json.loads(meta_file.read_text())
        authors   = entry["authors"]
        papers.append({
            "arxiv_id": entry["arxiv_id"],
            "title":    entry["title"],
            "authors":  ", ".join(authors[:3]) + (" et al." if len(authors) > 3 else ""),
            "abstract": metadata.get("abstract", ""),
            "text":     text,
        })
    return papers


papers = load_papers(CATALOG_PATH)

print(f"Loaded {len(papers)} papers:\n")
for p in papers:
    print(f"  [{p['arxiv_id']}]  {p['title']}")
    print(f"                   {p['authors']}")
    print(f"                   {len(p['text']):,} chars\n")

## Step 2 — Chunk Text

LLMs and embedding models have context limits.  We split each paper into overlapping windows so that:
- each chunk fits comfortably inside the embedding model's 512-token limit,
- adjacent chunks share an overlap region to prevent a sentence being cut across two chunks
  with neither side having full context.

`RecursiveCharacterTextSplitter` tries split points in this order:
`\n\n` → `\n` → `.` → ` ` → characters, keeping pieces as semantically coherent as possible.

In [ ]:
def chunk_papers(papers: list[dict]) -> list[dict]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = []
    for paper in papers:
        pieces = splitter.split_text(paper["text"])
        for i, piece in enumerate(pieces):
            chunks.append({
                "id":        f"{paper['arxiv_id']}_chunk_{i}",
                "text":      piece,
                "arxiv_id":  paper["arxiv_id"],
                "title":     paper["title"],
                "authors":   paper["authors"],
                "chunk_idx": i,
            })
    return chunks


chunks = chunk_papers(papers)

print(f"Total chunks : {len(chunks)}")
print(f"Avg length   : {sum(len(c['text']) for c in chunks) // len(chunks)} chars")
print(f"\nExample — paper 0, chunk 2:\n")
print(chunks[2]["text"])

## Step 3 — Embed & Store in ChromaDB

We use **`all-MiniLM-L6-v2`** — a fast, compact model (80 MB) that produces 384-dimensional embeddings.
ChromaDB wraps the model and handles the similarity index internally.

**The index is persisted to `./chroma_db/`**.  On re-runs, if the collection already has the correct
number of chunks, embedding is skipped automatically.

In [ ]:
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client     = chromadb.PersistentClient(path=str(CHROMA_PATH))
collection = client.get_or_create_collection(
    name=COLLECTION,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() == len(chunks):
    print(f"Collection already contains {collection.count()} chunks — skipping re-indexing.")
else:
    # Wipe any partial state before a fresh upsert
    existing = collection.get()["ids"]
    if existing:
        collection.delete(ids=existing)

    batch_size = 64
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start : start + batch_size]
        collection.add(
            ids       = [c["id"]   for c in batch],
            documents = [c["text"] for c in batch],
            metadatas = [
                {
                    "arxiv_id":  c["arxiv_id"],
                    "title":     c["title"],
                    "authors":   c["authors"],
                    "chunk_idx": c["chunk_idx"],
                }
                for c in batch
            ],
        )
        end = min(start + batch_size, len(chunks))
        print(f"  Indexed chunks {start}–{end - 1}  ({end}/{len(chunks)})")

    print(f"\nDone — {collection.count()} chunks stored in collection '{COLLECTION}'.")

## Step 4 — Retrieve Relevant Chunks

Given a user query, ChromaDB:
1. Embeds the query with the **same** `all-MiniLM-L6-v2` model used during indexing.
2. Computes cosine similarity between the query vector and every stored chunk vector.
3. Returns the **top-k** most similar chunks along with their metadata and distance scores.

ChromaDB returns *distances* (lower = more similar for cosine space), so we convert:
`similarity = 1 − distance`.

In [ ]:
def retrieve(query: str, n: int = TOP_K) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=n)
    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({
            "text":      doc,
            "title":     meta["title"],
            "arxiv_id":  meta["arxiv_id"],
            "authors":   meta["authors"],
            "chunk_idx": meta["chunk_idx"],
            "score":     round(1 - dist, 4),
        })
    return hits


# Quick test
test_query = "How does a spiking neural network work?"
hits = retrieve(test_query)

print(f"Query: {test_query}\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] score={h['score']}  [{h['arxiv_id']}] chunk {h['chunk_idx']}")
    print(f"     {h['title'][:65]}...")
    print(f"     {h['text'][:120]}...\n")

## Step 4b — Visualise Embeddings in 2-D

Each chunk is a 384-dimensional vector. We can't plot 384 dimensions, so we use **t-SNE** to project them down to 2D while preserving local neighbourhood structure.



In [ ]:
! uv pip install matplotlib

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer

VIZ_QUERY   = "How does MASS-RAG handle noisy retrieved contexts?"
SAMPLE_SIZE = 150
RANDOM_SEED = 42

model = SentenceTransformer("all-MiniLM-L6-v2")

random.seed(RANDOM_SEED)
sample_chunks = random.sample(chunks, min(SAMPLE_SIZE, len(chunks)))
retrieved      = retrieve(VIZ_QUERY, n=TOP_K)

all_texts = (
    [c["text"] for c in sample_chunks]
    + [h["text"] for h in retrieved]
    + [VIZ_QUERY]
)

print(f"Embedding {len(all_texts)} texts …")
embeddings = model.encode(all_texts, show_progress_bar=True)

n_bg  = len(sample_chunks)
n_ret = len(retrieved)

print("Running t-SNE …")
coords = TSNE(n_components=2, perplexity=30, random_state=RANDOM_SEED,
              max_iter=1000).fit_transform(embeddings)

xy_bg    = coords[:n_bg]
xy_ret   = coords[n_bg : n_bg + n_ret]
xy_query = coords[-1]

paper_ids = sorted({p["arxiv_id"] for p in papers})
cmap      = plt.cm.get_cmap("tab10", len(paper_ids))
pid2color = {pid: cmap(i) for i, pid in enumerate(paper_ids)}
bg_colors = [pid2color[c["arxiv_id"]] for c in sample_chunks]

fig, ax = plt.subplots(figsize=(13, 8))

ax.scatter(xy_bg[:, 0], xy_bg[:, 1],
           c=bg_colors, alpha=0.35, s=28, zorder=1)

for xy_r in xy_ret:
    ax.plot([xy_query[0], xy_r[0]], [xy_query[1], xy_r[1]],
            color="orange", alpha=0.6, linewidth=1.2, linestyle="--", zorder=2)

ax.scatter(xy_ret[:, 0], xy_ret[:, 1],
           c="gold", edgecolors="darkorange", linewidths=1.8,
           s=220, marker="*", zorder=3)

for rank, (xy_r, hit) in enumerate(zip(xy_ret, retrieved), 1):
    ax.annotate(
        f"#{rank}  {hit['arxiv_id']}\nscore={hit['score']}",
        xy=xy_r, xytext=(8, 4), textcoords="offset points",
        fontsize=7.5, color="saddlebrown",
        bbox=dict(boxstyle="round,pad=0.25", fc="lightyellow", alpha=0.85),
    )

ax.scatter(*xy_query, c="crimson", s=280, marker="X",
           edgecolors="darkred", linewidths=1.5, zorder=4)
ax.annotate("QUERY", xy=xy_query, xytext=(10, 5),
            textcoords="offset points",
            fontsize=8.5, fontweight="bold", color="darkred")

paper_legend = ax.legend(
    handles=[mpatches.Patch(color=pid2color[pid], label=pid, alpha=0.7)
             for pid in paper_ids],
    title="Paper (arXiv ID)", loc="upper left", fontsize=8,
)
ax.add_artist(paper_legend)

symbol_handles = [
    plt.Line2D([0], [0], marker="X", color="w", markerfacecolor="crimson",
               markersize=10, label="Query"),
    plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
               markeredgecolor="darkorange", markersize=12, label="Retrieved chunks"),
]
ax.legend(handles=symbol_handles, loc="lower right", fontsize=8)

ax.set_title(
    f't-SNE projection of {n_bg + n_ret + 1} chunk embeddings\n'
    f'Query: "{VIZ_QUERY[:70]}"',
    fontsize=11,
)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.grid(True, alpha=0.18)
plt.tight_layout()
plt.show()

## Step 5 — Generate Answer with Ollama

We build a **RAG prompt** that:
1. Provides the retrieved chunks as numbered context passages.
2. Instructs the model to answer **only** from the provided context and to cite the source.
3. Sends the prompt to the local Ollama server (`llama3.2:3b`) via the OpenAI-compatible endpoint.

> **Prerequisites**
> - Ollama running: `ollama serve`
> - Model pulled: `ollama pull llama3.2:3b`

#

### ΔΩΣΤΕ ΒΑΣΗ ΕΔΩ ΣΤΟ ΠΩΣ ΦΑΙΡΝΟΥΜΕ ΤΗΝ ΠΛΗΡΟΦΟΡΙΑ ΑΠΟ ΤΗΝ ΒΑΣΗ ΜΕΣΑ ΣΤΟ PROMPT

In [ ]:
llm = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")


def build_prompt(question: str, hits: list[dict]) -> str:
    passages = [
        f"[{i}] {h['title']} (arXiv:{h['arxiv_id']})\n{h['text']}"
        for i, h in enumerate(hits, 1)
    ]
    context = "\n\n".join(passages)
    return (
        "You are a research assistant. Answer the question using ONLY the context below.\n"
        "If the answer is not in the context, say: 'I cannot find this in the provided papers.'\n"
        "Cite the paper title or arXiv ID when you use information from it.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )


def rag_query(question: str, n: int = TOP_K) -> str:
    hits   = retrieve(question, n=n)
    prompt = build_prompt(question, hits)
    resp   = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content


print("rag_query() ready.")

## Demo — End-to-End Q&A

Three questions that span the 5 papers in our collection.

In [ ]:
q1 = "How do spiking neural networks differ from traditional artificial neural networks?"
print(f"Q: {q1}\n")
print(rag_query(q1))

## RAG vs No-RAG — Same Question

Ask the LLM the same question twice:
- **Without RAG**: no context is provided, the model answers from its training data alone.
- **With RAG**: the model receives the 4 most relevant paper chunks as context.

This shows how much grounding the retrieved context provides.

In [ ]:
def plain_query(question: str) -> str:
    """Ask the LLM directly, with no retrieved context."""
    resp = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
    )
    return resp.choices[0].message.content


compare_q = "How do spiking neural networks differ from traditional artificial neural networks?"

print("=" * 60)
print("WITHOUT RAG  (model answers from training data only)")
print("=" * 60)
print(plain_query(compare_q))

print("\n" + "=" * 60)
print("WITH RAG  (model receives relevant paper chunks as context)")
print("=" * 60)
print(rag_query(compare_q))

In [ ]:
q2 = "How does ArbGraph resolve conflicting evidence in retrieved documents?"
print(f"Q: {q2}\n")
print(rag_query(q2))

In [ ]:
q3 = "What role does vector memory play in WorldDB's architecture?"
print(f"Q: {q3}\n")
print(rag_query(q3))

## Try It Yourself

Change the question below and re-run the cell.

In [ ]:
my_question = "What datasets are used to evaluate RAG systems in these papers?"
print(f"Q: {my_question}\n")
print(rag_query(my_question))